# Single-Day Live Agent Simulation

Replays a past trading day using the **exact same** pre-filter + strategy stack as `run_live_daily.ipynb`.  
No API / KiteTicker needed — reads from parquet files only.

Steps:
1. Set `TARGET_DATE` below
2. Run all cells top-to-bottom
3. See which watchlist was selected, when a signal fired, and what the result would have been

In [ ]:
# ── SET THIS ──────────────────────────────────────────────────────────────────
TARGET_DATE = "2025-06-10"   # YYYY-MM-DD  — must exist in parquet files
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd
import json

trade_date = date.fromisoformat(TARGET_DATE)
print(f"Simulating live agent for: {trade_date}")

In [ ]:
# ── Cell 1: Load parquet history for all stocks ───────────────────────────────
from config.settings import STOCKS_DIR, INDEX_DIR, UNIVERSE_FILE, WEIGHTS_FILE
import pytz
IST = pytz.timezone("Asia/Kolkata")

universe = pd.read_csv(UNIVERSE_FILE)
symbols  = universe["tradingsymbol"].tolist()
print(f"Universe: {len(symbols)} stocks")

curr_yr = trade_date.year
prev_yr = curr_yr - 1

def _load_parquet_ist(path):
    """Read parquet and return datetime in IST naive (09:15, not 03:45)."""
    df = pd.read_parquet(path)
    dt = pd.to_datetime(df["datetime"])
    if dt.dt.tz is None:
        # Stored as naive — assume it's already IST wall time
        df["datetime"] = dt
    else:
        # Timezone-aware (+05:30 or UTC) → convert to IST → strip tz
        df["datetime"] = dt.dt.tz_convert(IST).dt.tz_localize(None)
    return df

all_history   = {}   # history BEFORE trade_date  (for pre-filter + strategy warmup)
today_candles = {}   # candles ON trade_date       (simulated live bars)
missing = []

for symbol in symbols:
    dfs = []
    for yr in [prev_yr, curr_yr]:
        p = STOCKS_DIR / str(yr) / f"{symbol}.parquet"
        if p.exists():
            dfs.append(_load_parquet_ist(p))
    if not dfs:
        missing.append(symbol)
        continue
    combined = (pd.concat(dfs).drop_duplicates("datetime")
                  .sort_values("datetime").reset_index(drop=True))
    hist  = combined[combined["datetime"].dt.date <  trade_date]
    today = combined[combined["datetime"].dt.date == trade_date]
    if not hist.empty:
        all_history[symbol] = hist.reset_index(drop=True)
    if not today.empty:
        today_candles[symbol] = today.reset_index(drop=True)

# NIFTY50 index
nifty_hist  = pd.DataFrame()
nifty_today = pd.DataFrame()
for yr in [prev_yr, curr_yr]:
    p = INDEX_DIR / str(yr) / "NIFTY50.parquet"
    if p.exists():
        df = _load_parquet_ist(p)
        nifty_hist  = pd.concat([nifty_hist,  df[df["datetime"].dt.date <  trade_date]])
        nifty_today = pd.concat([nifty_today, df[df["datetime"].dt.date == trade_date]])
nifty_hist  = nifty_hist.drop_duplicates("datetime").sort_values("datetime").reset_index(drop=True)
nifty_today = nifty_today.drop_duplicates("datetime").sort_values("datetime").reset_index(drop=True)

has_today = len([s for s in symbols if s in today_candles])
print(f"History loaded : {len(all_history)} stocks")
print(f"Today candles  : {has_today} stocks have data for {trade_date}")
print(f"Nifty today    : {len(nifty_today)} bars  "
      f"({nifty_today.iloc[0]['datetime'].strftime('%H:%M') if not nifty_today.empty else '-'}"
      f" → {nifty_today.iloc[-1]['datetime'].strftime('%H:%M') if not nifty_today.empty else '-'})")
if has_today < 100:
    print(f"\n⚠  WARNING: Only {has_today} stocks have candles for {trade_date}.")
    print("   Check that parquet files cover this date (run Section 6 in 01_setup_and_download.ipynb).")

In [ ]:
# ── Cell 2: Pre-market filter → watchlist (identical to live agent) ────────────
from watchlist.pre_filter import PreMarketFilter

pre_filter = PreMarketFilter()
watchlist  = pre_filter.build(trade_date, all_history)
wl_symbols = [e["symbol"] for e in watchlist]

print(f"\nWatchlist for {trade_date} ({len(wl_symbols)} stocks):")
print()
header = f"{'#':<4} {'Symbol':<14} {'Score':<7} {'Turnover(Cr)':<14} Signals"
print(header)
print("-" * 75)
for i, e in enumerate(watchlist, 1):
    sigs = ", ".join(s[0] for s in e["signals"])
    print(f"{i:<4} {e['symbol']:<14} {e['pre_score']:<7.2f} {e['turnover_cr']:<14.1f} {sigs}")

In [ ]:
# ── Cell 3: Build SimDataManager and load weights ─────────────────────────────

class SimDataManager:
    """Mimics LiveDataManager but serves parquet data bar-by-bar."""
    def __init__(self, symbols, history, today_candles, nifty_today_df, bar_idx=0):
        self.symbols       = symbols
        self._history      = history
        self._today        = today_candles   # full day candles per symbol
        self._nifty_today  = nifty_today_df
        self.bar_idx       = bar_idx         # how many bars closed so far today

    def get_history(self, symbol):
        return self._history.get(symbol, pd.DataFrame())

    def get_today(self, symbol):
        df = self._today.get(symbol, pd.DataFrame())
        return df.iloc[:self.bar_idx].reset_index(drop=True) if not df.empty else df

    def get_prev_day(self, symbol):
        hist = self._history.get(symbol)
        if hist is None or hist.empty:
            return None
        daily = (hist.groupby(hist["datetime"].dt.date)
                 .agg(open=("open","first"), high=("high","max"),
                      low=("low","min"),    close=("close","last"),
                      volume=("volume","sum")))
        return daily.iloc[-1] if not daily.empty else None

    def get_nifty_today(self):
        return self._nifty_today.iloc[:self.bar_idx].reset_index(drop=True)

    def get_last_price(self, symbol):
        """Close of the most recent bar — simulates the live tick price at bar close."""
        df = self._today.get(symbol, pd.DataFrame())
        if df.empty or self.bar_idx == 0:
            return None
        return float(df.iloc[min(self.bar_idx, len(df)) - 1]["close"])


# Load frozen weights
weights = {}
if WEIGHTS_FILE.exists():
    with open(WEIGHTS_FILE) as f:
        weights = json.load(f)
    print(f"Weights loaded: {len(weights)} strategies")
else:
    print("No weights file — using uniform weights")

# Build sim data manager with watchlist symbols only
sim_dm = SimDataManager(
    symbols      = wl_symbols,
    history      = {s: all_history[s] for s in wl_symbols if s in all_history},
    today_candles= {s: today_candles.get(s, pd.DataFrame()) for s in wl_symbols},
    nifty_today_df = nifty_today,
)

# Max bars available today (use the stock with most candles)
max_bars = max((len(today_candles.get(s, pd.DataFrame())) for s in wl_symbols), default=0)
print(f"Bars available for {trade_date}: {max_bars}  ({max_bars*5//60}h {max_bars*5%60}m of trading)")

In [ ]:
# ── Cell 4: Bar-by-bar simulation (same logic as live agent) ──────────────────
from live.live_engine import scan_once
from datetime import time as dtime

NO_ENTRY_AFTER = dtime(14, 0)
SQUARE_OFF     = dtime(15, 15)

trade_rec   = None
scan_log    = []

print(f"Scanning {len(wl_symbols)} watchlist stocks bar by bar...")
print()

for bar_idx in range(1, max_bars + 1):
    sim_dm.bar_idx = bar_idx

    # Get bar time from any watchlist stock with data
    bar_time = None
    for s in wl_symbols:
        df = today_candles.get(s, pd.DataFrame())
        if len(df) >= bar_idx:
            bar_time = pd.Timestamp(df.iloc[bar_idx - 1]["datetime"]).time()
            break
    if bar_time is None:
        continue
    if bar_time >= SQUARE_OFF:
        print(f"  {bar_time.strftime('%H:%M')} — 3:15 PM reached, stopping")
        break
    if bar_time >= NO_ENTRY_AFTER:
        scan_log.append({"bar": bar_time.strftime("%H:%M"), "result": "NO-ENTRY (after 2 PM)"})
        continue

    rec = scan_once(sim_dm, weights, trade_date)
    if rec:
        sig = rec["signal"]
        strategy_entry = sig.get("strategy_entry", sig["entry"])
        # Store when scan_once actually detected the signal (this bar's close)
        rec["detection_bar"] = bar_time.strftime("%H:%M")
        print(f"  {bar_time.strftime('%H:%M')} — SIGNAL: {rec['symbol']}")
        print(f"    Driver        : {sig['strategy']}")
        print(f"    Setup candle  : {sig['signal_time']}  (strategy condition formed on this bar)")
        print(f"    Strategy entry: Rs {strategy_entry:.2f}  (level from {sig['signal_time']} candle)")
        print(f"    Live entry    : Rs {sig['entry']:.2f}  (bar close at {bar_time.strftime('%H:%M')} — actual fill price)")
        print(f"    Target: {sig['target']:.2f}  Stop: {sig['stop']:.2f}  RR: {sig['rr']:.2f}")
        print(f"    Agreeing    : {rec['agreeing']}  Score: {rec['score']:.3f}  PredWin%: {rec['predicted_win_pct']:.1f}%")
        print(f"    Size        : Rs {rec['position_rs']:,.0f}  ({rec['shares']} shares)  [{rec['conviction_tier']}]")
        print(f"    Strategies fired: {rec['strategies_fired']}")
        trade_rec = rec
        scan_log.append({"bar": bar_time.strftime("%H:%M"), "result": f"SIGNAL → {rec['symbol']}"})
        break
    else:
        scan_log.append({"bar": bar_time.strftime("%H:%M"), "result": "no signal"})

if trade_rec is None:
    print("  No signal fired today — same result as live agent")

In [ ]:
# ── Cell 5: Simulate exit (target / stop / time exit) ────────────────────────
from backtester.cost_model import net_pnl as _net_pnl

if trade_rec is None:
    print("No trade to simulate exit for.")
else:
    sig    = trade_rec["signal"]
    symbol = trade_rec["symbol"]
    entry  = float(sig["entry"])           # live-price adjusted entry (bar close)
    target = float(sig["target"])
    stop   = float(sig["stop"])
    shares = int(trade_rec["shares"])
    strategy_entry = float(sig.get("strategy_entry", entry))

    setup_candle_time = sig["signal_time"]           # candle where strategy condition first matched
    detection_time    = trade_rec["detection_bar"]   # bar when scan_once() returned the signal

    today_df = today_candles.get(symbol, pd.DataFrame())

    # Exit scan starts from bars AFTER the detection bar (not after the setup candle)
    # Rationale: you can only enter after the detection bar closes; prior bars are pre-entry
    if not today_df.empty:
        after_entry = today_df[today_df["datetime"].dt.strftime("%H:%M") > detection_time]
    else:
        after_entry = pd.DataFrame()

    exit_price  = None
    exit_reason = "TIME_EXIT"
    exit_time   = "15:15"

    for _, bar in after_entry.iterrows():
        bar_t = pd.Timestamp(bar["datetime"]).strftime("%H:%M")
        if pd.Timestamp(bar["datetime"]).time() >= SQUARE_OFF:
            exit_price  = float(bar["open"])
            exit_reason = "TIME_EXIT"
            exit_time   = bar_t
            break
        if bar["high"] >= target:
            exit_price  = target
            exit_reason = "TARGET_HIT"
            exit_time   = bar_t
            break
        if bar["low"] <= stop:
            exit_price  = stop
            exit_reason = "STOP_HIT"
            exit_time   = bar_t
            break

    if exit_price is None:
        exit_price = float(today_df.iloc[-1]["close"]) if not today_df.empty else entry
        exit_reason = "TIME_EXIT (last bar)"
        exit_time = today_df.iloc[-1]["datetime"].strftime("%H:%M") if not today_df.empty else "15:15"

    pnl     = round(_net_pnl(entry, exit_price, shares), 2)
    pnl_pct = round((exit_price - entry) / entry * 100, 2)
    result  = "EXACT_WIN" if exit_reason == "TARGET_HIT" else ("WIN" if pnl > 0 else "LOSS")

    print("=" * 60)
    print(f"  TRADE RESULT — {trade_date}")
    print("=" * 60)
    print(f"  Symbol        : {symbol}")
    print(f"  Strategy      : {sig['strategy']}")
    print(f"  Setup candle  : {setup_candle_time}  ({sig['strategy']} condition formed)")
    print(f"  Detected      : {detection_time}  (scan — entry placed after this bar)")
    print(f"  Strategy entry: Rs {strategy_entry:.2f}  (level from {setup_candle_time} candle)")
    print(f"  Live entry    : Rs {entry:.2f}  (bar close at {detection_time} — actual fill)")
    print(f"  Target        : Rs {target:.2f}")
    print(f"  Stop          : Rs {stop:.2f}")
    print(f"  Exit          : Rs {exit_price:.2f}  @ {exit_time}  ({exit_reason})")
    print(f"  Shares        : {shares}  |  Size: Rs {trade_rec['position_rs']:,.0f}")
    print(f"  P&L (net)     : Rs {pnl:+,.2f}  ({pnl_pct:+.2f}% gross, after all costs)")
    print(f"  Result        : {result}")
    print("=" * 60)
    if strategy_entry != entry:
        drift_pct = (entry - strategy_entry) / strategy_entry * 100
        print()
        print(f"  NOTE: Strategy level was Rs {strategy_entry:.2f} but stock had moved to")
        print(f"  Rs {entry:.2f} ({drift_pct:+.2f}%) by detection time {detection_time}.")
        print(f"  Stop/target distances preserved — R:R ratio unchanged.")

In [ ]:
# ── Cell 6: Bar-by-bar scan log ───────────────────────────────────────────────
import pandas as pd
if scan_log:
    print(f"Bar-by-bar scan log ({len(scan_log)} bars checked):")
    print()
    for entry in scan_log:
        print(f"  {entry['bar']}  {entry['result']}")
else:
    print("No scan log — no bars were processed.")